In [ ]:
!pip install -q x-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 2.2 MB/s eta 0:00:00


In [ ]:
# @title 🛠️ Appendix Physical Validation (Gain & Stability)
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer
import sys
import os

# ==============================================================================
# 1. SETUP & MODEL LOADING
# ==============================================================================
REPO_ID = "prism-lab/prism-shimmer-100k"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"⚙️ Hardware: {DEVICE}")
print(f"📥 Loading PRISM from {REPO_ID}...")

# Download architecture
os.makedirs("shimmer_code", exist_ok=True)
hf_hub_download(repo_id=REPO_ID, filename="modeling_prism_gated.py", local_dir="shimmer_code")
sys.path.append("shimmer_code")

from modeling_prism_gated import PRISMHybrid_RoPE

# Load Model
tokenizer = AutoTokenizer.from_pretrained(REPO_ID)
CONFIG = {
    "vocab_size": 58101, "d_model": 512, "num_heads": 8, "dff": 2048,
    "dropout": 0.1, "max_length": 128, "num_encoder_layers": 6,
    "num_refining_layers": 0, "num_decoder_layers": 6
}
model = PRISMHybrid_RoPE(**CONFIG)
state_dict = torch.load(hf_hub_download(repo_id=REPO_ID, filename="pytorch_model.bin"), map_location=DEVICE)
model.load_state_dict(state_dict)
model.to(DEVICE)
model.eval()

print("✅ Model Ready.")

# ==============================================================================
# 2. DATASETS (Placeholders)
# ==============================================================================
# ⚠️ PASTE YOUR FULL LISTS HERE FROM THE PREVIOUS STEP
# N=76 Hard, N=70 Easy

raw_poly_candidates = [
    # --- ORIGINAL SET ---
    ("Ich gehe zur Bank um Geld zu holen", "Bank"), ("Die Bank hat hohe Zinsen", "Bank"),
    ("Wir saßen auf einer Bank im Park", "Bank"), ("Die Bank aus Holz war bequem", "Bank"),
    ("Das Schloss hat viele Türme", "Schloss"), ("Der König wohnt im Schloss", "Schloss"),
    ("Der Schlüssel steckt im Schloss", "Schloss"), ("Das Schloss an der Tür klemmt", "Schloss"),
    ("Der Leiter der Firma ist streng", "Leiter"), ("Unser Leiter plant das Projekt", "Leiter"),
    ("Ich steige auf die Leiter", "Leiter"), ("Die Leiter ist aus Aluminium", "Leiter"),
    ("Die Lampe hängt an der Decke", "Decke"), ("Die Decke ist weiß gestrichen", "Decke"),
    ("Mir ist kalt gib mir eine Decke", "Decke"), ("Die Decke aus Wolle ist warm", "Decke"),
    ("Der Kiefer ist ein Nadelbaum", "Kiefer"), ("Das Holz der Kiefer ist weich", "Kiefer"),
    ("Der Arzt röntgt meinen Kiefer", "Kiefer"), ("Er hat Schmerzen im Kiefer", "Kiefer"),
    ("Der Strauß ist ein schneller Vogel", "Strauß"), ("Dieser Strauß kann nicht fliegen", "Strauß"),
    ("Sie kaufte einen bunten Strauß", "Strauß"), ("Der Strauß Blumen duftet gut", "Strauß"),
    ("Er schoss ein schönes Tor", "Tor"), ("Der Ball flog ins Tor", "Tor"),
    ("Das eiserne Tor war verschlossen", "Tor"), ("Sie öffneten das große Tor", "Tor"),
    ("Wir tanzen auf dem Ball", "Ball"), ("Der Maskenball war elegant", "Ball"),
    ("Er warf den Ball weit weg", "Ball"), ("Der Ball ist rund und rot", "Ball"),
    ("Die Schlange im Zoo ist giftig", "Schlange"), ("Die Schlange zischte laut", "Schlange"),
    ("Wir stehen in einer langen Schlange", "Schlange"), ("Die Schlange an der Kasse war lang", "Schlange"),
    ("Der Strom ist ausgefallen", "Strom"), ("Strom kostet viel Geld", "Strom"),
    ("Der Strom fließt ins Meer", "Strom"), ("Wir schwammen gegen den Strom", "Strom"),
    ("Seine Mutter ist sehr nett", "Mutter"), ("Die Mutter kocht das Essen", "Mutter"),
    ("Die Mutter passt auf die Schraube", "Mutter"), ("Ich brauche eine neue Mutter", "Mutter"),
    ("Die Birne schmeckt süß", "Birne"), ("Ich esse gerne eine Birne", "Birne"),
    ("Die Birne in der Lampe ist kaputt", "Birne"), ("Wir müssen die Birne wechseln", "Birne"),
    # --- EXPANSION SET ---
    ("Das Gericht hat ihn verurteilt", "Gericht"), ("Der Anwalt geht zum Gericht", "Gericht"),
    ("Mein Lieblingsessen ist ein Gericht aus Reis", "Gericht"), ("Das Gericht schmeckt sehr salzig", "Gericht"),
    ("Der Ton war sehr laut", "Ton"), ("Ich hörte einen hohen Ton", "Ton"),
    ("Die Vase ist aus Ton", "Ton"), ("Wir formen Figuren aus Ton", "Ton"),
    ("Das Blatt fällt vom Baum", "Blatt"), ("Im Herbst werden die Blätter braun", "Blatt"),
    ("Ich schreibe auf ein Blatt Papier", "Blatt"), ("Gib mir bitte ein leeres Blatt", "Blatt"),
    ("Der Nagel steckt in der Wand", "Nagel"), ("Ich schlage den Nagel mit dem Hammer", "Nagel"),
    ("Mein Nagel ist abgebrochen", "Nagel"), ("Sie lackiert sich den Nagel rot", "Nagel"),
    ("Die Maus frisst den Käse", "Maus"), ("Die Katze jagt die Maus", "Maus"),
    ("Ich klicke mit der Maus", "Maus"), ("Der Computer braucht eine neue Maus", "Maus"),
    ("Die Erde dreht sich um die Sonne", "Erde"), ("Der Astronaut schaut auf die Erde", "Erde"),
    ("Die Blume braucht frische Erde", "Erde"), ("Er gräbt ein Loch in die Erde", "Erde"),
    ("Der Hahn kräht am Morgen", "Hahn"), ("Der Hahn hat bunte Federn", "Hahn"),
    ("Der Wasserhahn tropft", "Hahn"), ("Dreh bitte den Hahn zu", "Hahn"),
    ("Die Schale der Orange ist bitter", "Schale"), ("Er wirft die Schale weg", "Schale"),
    ("Die Schale steht auf dem Tisch", "Schale"), ("Ich esse Müsli aus der Schale", "Schale"),
    ("Der Bauer melkt die Kühe", "Bauer"), ("Der Bauer fährt auf dem Traktor", "Bauer"),
    ("Ich ziehe den Bauer auf E4", "Bauer"), ("Der Bauer schlägt den Turm", "Bauer"),
]

# B. EASY MODE (Casual)
raw_casual_candidates = [
    ("Die Katze schläft", "Katze"), ("Der Hund bellt", "Hund"), ("Das Auto fährt", "Auto"),
    ("Wasser ist nass", "Wasser"), ("Das Brot schmeckt gut", "Brot"), ("Die Sonne scheint", "Sonne"),
    ("Der Mond leuchtet", "Mond"), ("Das Buch ist spannend", "Buch"), ("Der Tisch ist rund", "Tisch"),
    ("Der Stuhl ist bequem", "Stuhl"), ("Der Apfel ist rot", "Apfel"), ("Meine Hand ist kalt", "Hand"),
    ("Das Herz klopft", "Herz"), ("Wir haben Zeit", "Zeit"), ("Geld ist wichtig", "Geld"),
    ("Musik ist schön", "Musik"), ("Der Film ist zu Ende", "Film"), ("Das Spiel beginnt", "Spiel"),
    ("Die Schule ist aus", "Schule"), ("Die Stadt ist laut", "Stadt"), ("Der Fluss fließt", "Fluss"),
    ("Das Meer ist tief", "Meer"), ("Kaffee ist schwarz", "Kaffee"), ("Milch ist weiß", "Milch"),
    ("Der Bruder lacht", "Bruder"), ("Die Schwester weint", "Schwester"), ("Das Haus ist groß", "Haus"),
    ("Der Garten ist grün", "Garten"), ("Der Sommer ist heiß", "Sommer"), ("Der Winter ist kalt", "Winter"),
    ("Das Fenster ist offen", "Fenster"), ("Die Tür ist zu", "Tür"), ("Der Boden ist sauber", "Boden"),
    ("Die Wand ist weiß", "Wand"), ("Das Dach ist rot", "Dach"), ("Der Wald ist dunkel", "Wald"),
    ("Der Berg ist hoch", "Berg"), ("Der See ist ruhig", "See"), ("Das Tier ist wild", "Tier"),
    ("Der Mensch denkt", "Mensch"), ("Das Kind spielt", "Kind"), ("Die Frau arbeitet", "Frau"),
    ("Der Mann schläft", "Mann"), ("Das Auge sieht", "Auge"), ("Das Ohr hört", "Ohr"),
    ("Die Nase riecht", "Nase"), ("Der Mund spricht", "Mund"), ("Der Arm ist stark", "Arm"),
    ("Das Bein tut weh", "Bein"), ("Der Fuß ist groß", "Fuß"), ("Der Tee ist heiß", "Tee"),
    ("Das Bier ist kalt", "Bier"), ("Der Wein ist rot", "Wein"), ("Das Glas ist voll", "Glas"),
    ("Die Tasse ist leer", "Tasse"), ("Der Teller ist blau", "Teller"), ("Die Gabel ist spitz", "Gabel"),
    ("Der Löffel ist rund", "Löffel"), ("Das Messer ist scharf", "Messer"), ("Der Stift schreibt", "Stift"),
    ("Der Brief ist lang", "Brief"), ("Das Bild ist schön", "Bild"), ("Die Uhr tickt", "Uhr"),
    ("Das Bett ist weich", "Bett"), ("Der Schrank ist voll", "Schrank"), ("Das Sofa ist neu", "Sofa"),
    ("Das Radio spielt", "Radio"), ("Das Jahr ist um", "Jahr"), ("Der Tag war lang", "Tag"),
    ("Die Nacht ist kurz", "Nacht")
]

# ==============================================================================
# 3. HELPER: Single-Token Validator
# ==============================================================================
def filter_dataset(candidates, tokenizer, label):
    valid = []
    for ctx, tgt in candidates:
        t1 = tokenizer.encode(tgt, add_special_tokens=False)
        t2 = tokenizer.encode(" " + tgt, add_special_tokens=False)
        if len(t1) == 1 or len(t2) == 1: valid.append((ctx, tgt))
    print(f"✅ {label}: {len(valid)} atomic examples validated.")
    return valid

def find_token_index(input_ids, target_word, tokenizer):
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    for i, t in enumerate(tokens):
        clean = t.replace('Ġ', '').replace('▁', '').replace(' ', '')
        if target_word.lower() == clean.lower(): return i
    for i, t in enumerate(tokens): # Fallback
        clean = t.replace('Ġ', '').replace('▁', '').replace(' ', '')
        if target_word.lower() in clean.lower(): return i
    return 1

# ==============================================================================
# 4. PHYSICAL PROBE (Gain & Magnitude)
# ==============================================================================
def run_physical_probe(model, tokenizer, dataset, label, device):
    """
    Extracts Gain (Ratio) and Raw Magnitude (Norm) for CV analysis.
    """
    num_layers = len(model.prism_encoder.layers)

    # Store Gain (for Fig B3) and Magnitude (for Fig B1)
    gain_stats = {i: [] for i in range(num_layers)}
    magnitude_stats = {i: [] for i in range(num_layers)}
    embedding_mags = []

    hook_data = {}

    def physics_hook(layer_idx):
        def hook(module, input, output):
            x, y = input[0].detach(), output.detach()

            # 1. Norms (Energy)
            norm_x = torch.norm(x, p=2, dim=-1)
            norm_y = torch.norm(y, p=2, dim=-1)

            # 2. Gain Calculation
            gain = norm_y / (norm_x + 1e-9)

            hook_data[f'layer_{layer_idx}'] = {
                'gain': gain.cpu(),
                'mag': norm_y.cpu() # Output magnitude
            }
        return hook

    # Register Hooks
    model.prism_encoder.apply(lambda m: m._forward_hooks.clear())
    for i, layer in enumerate(model.prism_encoder.layers):
        layer.register_forward_hook(physics_hook(i))

    # Run Probe
    print(f"🔬 Measuring Physics on {len(dataset)} {label} examples...")
    for context, target in dataset:
        hook_data = {}
        inputs = tokenizer(context, return_tensors="pt").to(device)

        with torch.no_grad():
            # Capture embedding magnitude before encoder
            emb = model.harmonic_embedding(inputs.input_ids)
            embedding_mags.append(torch.norm(emb, p=2, dim=-1).flatten().cpu())

            # Forward pass
            src_mask = (inputs.input_ids == tokenizer.pad_token_id)
            model.prism_encoder(emb, src_mask)

        idx = find_token_index(inputs.input_ids[0], target, tokenizer)

        for i in range(num_layers):
            if f'layer_{i}' in hook_data:
                data = hook_data[f'layer_{i}']

                # Extract atomic token metrics
                g = data['gain']
                m = data['mag']

                val_g = g[0, idx].item() if g.dim() > 1 else g[idx].item()
                val_m = m[0, idx].item() if m.dim() > 1 else m[idx].item()

                gain_stats[i].append(val_g)
                magnitude_stats[i].append(val_m)

    model.prism_encoder.apply(lambda m: m._forward_hooks.clear())

    return {
        'gain': pd.DataFrame(gain_stats),
        'magnitude': magnitude_stats, # Dict of lists
        'embedding': torch.cat(embedding_mags).numpy()
    }

# ==============================================================================
# 5. EXECUTION
# ==============================================================================
# Filter
ds_hard = filter_dataset(raw_poly_candidates, tokenizer, "HARD")
ds_easy = filter_dataset(raw_casual_candidates, tokenizer, "EASY")

# Run
res_hard = run_physical_probe(model, tokenizer, ds_hard, "HARD", DEVICE)
res_easy = run_physical_probe(model, tokenizer, ds_easy, "EASY", DEVICE)

# ==============================================================================
# 6. PLOT FIGURE B3: ISO-ENERGETIC GAIN
# ==============================================================================
def plot_gain_chart(res_hard, res_easy):
    df_h = res_hard['gain']
    df_e = res_easy['gain']

    layers = list(df_h.columns)
    means_h = [df_h[i].mean() for i in layers]
    stds_h = [df_h[i].std() for i in layers]
    means_e = [df_e[i].mean() for i in layers]
    stds_e = [df_e[i].std() for i in layers]

    x = np.arange(len(layers))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 4), dpi=300)
    ax.bar(x - width/2, means_h, width, yerr=stds_h, label='Ambiguous',
           color='indianred', alpha=0.8, capsize=3)
    ax.bar(x + width/2, means_e, width, yerr=stds_e, label='Unambiguous',
           color='steelblue', alpha=0.8, capsize=3)

    ax.axhline(y=1.0, color='black', linestyle='--', linewidth=2, label='Unity Gain (g=1.0)')
    ax.set_ylabel('Signal Gain (||y|| / ||x||)', fontweight='bold')
    ax.set_xlabel('Layer Depth')
    ax.set_xticks(x)
    ax.set_xticklabels(layers)
    ax.set_ylim(0.85, 1.15) # Zoom in to show it's flat
    ax.legend(loc='upper right')
    ax.set_title('Iso-Energetic Constraint: Gain ≈ 1.0 Across All Conditions', fontweight='bold')
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    plt.tight_layout()
    plt.savefig("fig_B3_gain.png")
    plt.show()
    print("✅ Figure B3 Saved.")

# ==============================================================================
# 7. PLOT FIGURE B1: MAGNITUDE STABILITY (CV)
# ==============================================================================
def plot_cv_chart(res_hard, res_easy):
    # Combine data to check global network stability
    # CV = sigma / mu

    stages = ["Embedding"]
    cvs = []

    # 1. Embedding Stage
    all_emb = np.concatenate([res_hard['embedding'], res_easy['embedding']])
    cvs.append(all_emb.std() / all_emb.mean())

    # 2. Layers 0-5
    for i in range(6):
        # Flatten lists
        mags_h = np.array(res_hard['magnitude'][i])
        mags_e = np.array(res_easy['magnitude'][i])
        all_mags = np.concatenate([mags_h, mags_e])

        cv = all_mags.std() / (all_mags.mean() + 1e-9)
        cvs.append(cv)
        stages.append(f"Layer {i}")

    mean_cv = np.mean(cvs)

    fig, ax = plt.subplots(figsize=(8, 4), dpi=300)
    bars = ax.bar(stages, cvs, color='steelblue', alpha=0.8, edgecolor='grey')

    ax.axhline(y=mean_cv, color='red', linestyle='--', label=f'Mean CV = {mean_cv:.3f}')
    ax.set_ylabel('Coefficient of Variation (σ/μ)', fontweight='bold')
    ax.set_xlabel('Network Stage')
    ax.set_title('Magnitude Stability Across Layers (Iso-Energetic Check)', fontweight='bold')
    ax.set_ylim(0, 1.0)
    ax.legend()

    # Label bars
    for bar, v in zip(bars, cvs):
        ax.text(bar.get_x() + bar.get_width()/2, v, f"{v:.3f}",
                ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig("fig_B1_cv.png")
    plt.show()
    print("✅ Figure B1 Saved.")

# ==============================================================================
# RUN PLOTS
# ==============================================================================
plot_gain_chart(res_hard, res_easy)
plot_cv_chart(res_hard, res_easy)